# Access Control: Roles and Privileges

## Overview
PostgreSQL uses Role-Based Access Control (RBAC). Roles define who can do what — which tables can be read, written, or modified. Good access control follows the principle of least privilege: every role gets only the minimum access required for its function.

**Role design pattern:**
```
Functional roles (no login):  clinician_role, analyst_role, admin_role, auditor_role
     ↓  GRANT role TO user
Login users:                  dr_lee, analyst1, admin1, auditor1
```

**Dataset:** healthcare schema with users, patients, patient_records, and a role_permissions table modelling the intended RBAC matrix for 5 roles across 2 tables, including row-filter expressions that will become RLS policies.

---

In [1]:
import sqlite3, hashlib, secrets, base64, hmac
from datetime import datetime

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE users (
    user_id TEXT PRIMARY KEY, username TEXT NOT NULL UNIQUE,
    role_name TEXT NOT NULL, department TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, full_name TEXT NOT NULL,
    dob TEXT NOT NULL, province TEXT NOT NULL, assigned_clinician TEXT
);
CREATE TABLE patient_records (
    record_id TEXT PRIMARY KEY, patient_id TEXT NOT NULL,
    record_type TEXT NOT NULL, content TEXT NOT NULL,
    sensitivity TEXT DEFAULT 'normal', created_by TEXT NOT NULL, created_at TEXT
);
CREATE TABLE financial_accounts (
    account_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL,
    account_type TEXT NOT NULL, balance_enc TEXT, ssn_hash TEXT, created_at TEXT
);
CREATE TABLE audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT, event_time TEXT DEFAULT (datetime('now')),
    username TEXT NOT NULL, action TEXT NOT NULL, table_name TEXT,
    record_id TEXT, old_value TEXT, new_value TEXT, ip_address TEXT, success INTEGER DEFAULT 1
);
CREATE TABLE role_permissions (
    role_name TEXT NOT NULL, table_name TEXT NOT NULL,
    can_select INTEGER DEFAULT 0, can_insert INTEGER DEFAULT 0,
    can_update INTEGER DEFAULT 0, can_delete INTEGER DEFAULT 0,
    row_filter TEXT, PRIMARY KEY (role_name, table_name)
);
""")

conn.executemany("INSERT INTO users VALUES (?,?,?,?,?)", [
    ('U001','dr.lee',    'clinician','Cardiology',    1),
    ('U002','dr.pham',   'clinician','Endocrinology',1),
    ('U003','analyst1',  'analyst',  'Reporting',    1),
    ('U004','admin1',    'admin',    'IT',           1),
    ('U005','auditor1',  'auditor',  'Compliance',   1),
    ('U006','patient_p001','patient',None,            1),
])
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", [
    ('P001','Alice Ngata',  '1980-03-15','NB','dr.lee'),
    ('P002','Bob Chen',     '1972-07-22','ON','dr.pham'),
    ('P003','Fatima Rashid','1986-11-05','BC','dr.lee'),
    ('P004','James MacLeod','1963-01-30','NS','dr.pham'),
    ('P005','Mei-Ling Tran','1966-08-19','QC','dr.lee'),
])
conn.executemany("INSERT INTO patient_records VALUES (?,?,?,?,?,?,?)", [
    ('R001','P001','diagnosis',  'Hypertension, Stage 2',      'normal',    'dr.lee', '2024-01-15'),
    ('R002','P001','prescription','Lisinopril 10mg daily',     'normal',    'dr.lee', '2024-01-15'),
    ('R003','P002','diagnosis',  'Type 2 Diabetes',            'normal',    'dr.pham','2024-01-10'),
    ('R004','P002','lab',        'HbA1c 8.4%',                 'normal',    'dr.pham','2024-01-10'),
    ('R005','P003','diagnosis',  'Asthma, moderate persistent','sensitive', 'dr.lee', '2024-02-01'),
    ('R006','P004','diagnosis',  'CKD Stage 3, T2DM',          'sensitive', 'dr.pham','2024-02-15'),
    ('R007','P005','prescription','Insulin glargine 20u',      'restricted','dr.lee', '2024-03-01'),
])
salt = secrets.token_hex(8)
def hash_ssn(ssn): return hashlib.sha256((salt+ssn).encode()).hexdigest()
conn.executemany("INSERT INTO financial_accounts VALUES (?,?,?,?,?,datetime('now'))", [
    ('ACC001','P001','chequing','[encrypted:AES256]',hash_ssn('123-45-6789')),
    ('ACC002','P002','savings', '[encrypted:AES256]',hash_ssn('987-65-4321')),
    ('ACC003','P003','chequing','[encrypted:AES256]',hash_ssn('456-78-9012')),
])
conn.executemany("INSERT INTO role_permissions VALUES (?,?,?,?,?,?,?)", [
    ('clinician','patients',       1,0,1,0,'assigned_clinician = current_user'),
    ('clinician','patient_records',1,1,1,0,'patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician = current_user)'),
    ('analyst',  'patients',       1,0,0,0,'province IS NOT NULL'),
    ('analyst',  'patient_records',1,0,0,0,"sensitivity = 'normal'"),
    ('admin',    'patients',       1,1,1,1,None),
    ('admin',    'patient_records',1,1,1,1,None),
    ('auditor',  'audit_log',      1,0,0,0,None),
    ('auditor',  'patients',       1,0,0,0,None),
    ('patient',  'patients',       1,0,0,0,'patient_id = current_patient_id()'),
    ('patient',  'patient_records',1,0,0,0,'patient_id = current_patient_id()'),
])
conn.executemany(
    "INSERT INTO audit_log (username,action,table_name,record_id,old_value,new_value,ip_address,success) VALUES (?,?,?,?,?,?,?,?)", [
    ('dr.lee',  'SELECT','patient_records','R001',None,None,'10.0.1.5',1),
    ('dr.lee',  'UPDATE','patient_records','R002','{"content":"Lisinopril 5mg"}','{"content":"Lisinopril 10mg"}','10.0.1.5',1),
    ('analyst1','SELECT','patients',       None,  None,None,'10.0.2.8',1),
    ('analyst1','EXPORT','patient_records',None,  None,None,'10.0.2.8',1),
    ('unknown', 'LOGIN', None,             None,  None,None,'203.0.113.9',0),
    ('dr.pham', 'SELECT','patient_records','R005',None,None,'10.0.1.6',0),
    ('admin1',  'DELETE','patient_records','R007',None,None,'10.0.1.1',1),
])
conn.commit()
print("Dataset loaded:")
for t in ['users','patients','patient_records','financial_accounts','audit_log','role_permissions']:
    print(f"  {t}: {conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]} rows")

print("=== PostgreSQL RBAC: 3-step pattern ===")
print()
print("""
-- Step 1: Create functional roles (no login)
CREATE ROLE clinician_role;
CREATE ROLE analyst_role;
CREATE ROLE admin_role;
CREATE ROLE auditor_role;

-- Step 2: Grant table privileges to roles
GRANT SELECT, INSERT, UPDATE ON patient_records TO clinician_role;
GRANT SELECT                 ON patient_records TO analyst_role;
GRANT ALL PRIVILEGES         ON patient_records TO admin_role;
GRANT SELECT                 ON audit_log       TO auditor_role;

-- Step 3: Create login users and assign roles
CREATE USER dr_lee WITH PASSWORD '...' LOGIN;
GRANT clinician_role TO dr_lee;

-- Revoke default open access (always do this after DB creation):
REVOKE ALL ON ALL TABLES IN SCHEMA public FROM PUBLIC;
REVOKE CREATE ON SCHEMA public FROM PUBLIC;

-- Set default privileges for future tables:
ALTER DEFAULT PRIVILEGES IN SCHEMA public
    GRANT SELECT ON TABLES TO analyst_role;
""")

print("Role permission matrix:")
rows = conn.execute("""
    SELECT role_name, table_name,
           CASE can_select WHEN 1 THEN 'Y' ELSE '-' END AS sel,
           CASE can_insert WHEN 1 THEN 'Y' ELSE '-' END AS ins,
           CASE can_update WHEN 1 THEN 'Y' ELSE '-' END AS upd,
           CASE can_delete WHEN 1 THEN 'Y' ELSE '-' END AS del,
           COALESCE(row_filter,'(none)') AS row_filter
    FROM role_permissions ORDER BY role_name, table_name
""").fetchall()
print(f"  {'role':<12s}  {'table':<16s}  SEL  INS  UPD  DEL  row_filter")
print("  " + "-"*80)
for r in rows:
    print(f"  {r['role_name']:<12s}  {r['table_name']:<16s}  {r['sel']}    {r['ins']}    {r['upd']}    {r['del']}    {r['row_filter']}")


Dataset loaded:
  users: 6 rows
  patients: 5 rows
  patient_records: 7 rows
  financial_accounts: 3 rows
  audit_log: 7 rows
  role_permissions: 10 rows
=== PostgreSQL RBAC: 3-step pattern ===


-- Step 1: Create functional roles (no login)
CREATE ROLE clinician_role;
CREATE ROLE analyst_role;
CREATE ROLE admin_role;
CREATE ROLE auditor_role;

-- Step 2: Grant table privileges to roles
GRANT SELECT, INSERT, UPDATE ON patient_records TO clinician_role;
GRANT SELECT                 ON patient_records TO analyst_role;
GRANT ALL PRIVILEGES         ON patient_records TO admin_role;
GRANT SELECT                 ON audit_log       TO auditor_role;

-- Step 3: Create login users and assign roles
CREATE USER dr_lee WITH PASSWORD '...' LOGIN;
GRANT clinician_role TO dr_lee;

-- Revoke default open access (always do this after DB creation):
REVOKE ALL ON ALL TABLES IN SCHEMA public FROM PUBLIC;
REVOKE CREATE ON SCHEMA public FROM PUBLIC;

-- Set default privileges for future tables:
ALTER DEFAUL

In [2]:

print("=== Least privilege violations and fixes ===")
violations = [
    ("analyst has DELETE on patient_records", "Analysts only need SELECT",
     "REVOKE DELETE ON patient_records FROM analyst_role"),
    ("App connects as postgres (superuser)",  "Superuser bypasses all security",
     "CREATE ROLE app_user LOGIN; GRANT SELECT,INSERT,UPDATE ON ... TO app_user"),
    ("GRANT ALL instead of specific privs",   "Gives TRUNCATE, DROP, etc. unintentionally",
     "GRANT SELECT, INSERT, UPDATE ON patients TO clinician_role"),
    ("No RLS — clinicians see all patients",  "Clinician should see only their own patients",
     "ENABLE ROW LEVEL SECURITY + CREATE POLICY clinician_own_patients"),
    ("PUBLIC has CREATE on public schema",    "Any user can create objects in public schema",
     "REVOKE CREATE ON SCHEMA public FROM PUBLIC"),
]
for v, why, fix in violations:
    print(f"  Violation: {v}")
    print(f"  Why bad:   {why}")
    print(f"  Fix:       {fix}")
    print()

print("=== Schema separation: secure view for analysts ===")
analyst_view = conn.execute("""
    SELECT patient_id,
           CASE WHEN CAST(SUBSTR(dob,1,4) AS INT) < 1960 THEN '60+'
                WHEN CAST(SUBSTR(dob,1,4) AS INT) < 1980 THEN '40-59'
                ELSE '20-39' END AS age_group,
           province, 'REDACTED' AS full_name
    FROM patients
""").fetchall()
print("analytics.patients_anon view (analyst sees this — no names or exact DOBs):")
print(f"  {'patient_id':<12s}  {'age_group':<10s}  {'province':<10s}  full_name")
print("  " + "-"*48)
for r in analyst_view:
    print(f"  {r[0]:<12s}  {r[1]:<10s}  {r[2]:<10s}  {r[3]}")
print()
clinician_view = conn.execute(
    "SELECT patient_id,full_name,dob,province FROM patients WHERE assigned_clinician='dr.lee'"
).fetchall()
print("dr.lee sees (own patients only via RLS):")
for r in clinician_view: print(f"  {r[0]}  {r[1]:<16s}  {r[2]}  {r[3]}")


=== Least privilege violations and fixes ===
  Violation: analyst has DELETE on patient_records
  Why bad:   Analysts only need SELECT
  Fix:       REVOKE DELETE ON patient_records FROM analyst_role

  Violation: App connects as postgres (superuser)
  Why bad:   Superuser bypasses all security
  Fix:       CREATE ROLE app_user LOGIN; GRANT SELECT,INSERT,UPDATE ON ... TO app_user

  Violation: GRANT ALL instead of specific privs
  Why bad:   Gives TRUNCATE, DROP, etc. unintentionally
  Fix:       GRANT SELECT, INSERT, UPDATE ON patients TO clinician_role

  Violation: No RLS — clinicians see all patients
  Why bad:   Clinician should see only their own patients
  Fix:       ENABLE ROW LEVEL SECURITY + CREATE POLICY clinician_own_patients

  Violation: PUBLIC has CREATE on public schema
  Why bad:   Any user can create objects in public schema
  Fix:       REVOKE CREATE ON SCHEMA public FROM PUBLIC

=== Schema separation: secure view for analysts ===
analytics.patients_anon view (analyst

---
## Common Pitfalls

**1. Connecting the application as superuser or postgres**
If the app connects as `postgres`, any SQL injection gives the attacker full superuser access — including `COPY TO/FROM`, `CREATE EXTENSION`, and every table in every database. Create a dedicated application role with only the minimum required privileges.

**2. Leaving PUBLIC privileges intact**
By default PostgreSQL grants `CREATE` on the public schema and `CONNECT` on all databases to `PUBLIC`. Run `REVOKE ALL ON ALL TABLES IN SCHEMA public FROM PUBLIC` and `REVOKE CREATE ON SCHEMA public FROM PUBLIC` immediately after database creation.

**3. Granting privileges directly to users instead of roles**
`GRANT SELECT ON patients TO analyst1` ties a privilege to one person. When analyst1 leaves, you must track down individual grants. Grant to roles, assign users to roles, manage access entirely through role membership.

**4. Using GRANT ALL instead of specific privileges**
`GRANT ALL ON patient_records TO analyst_role` gives analysts DELETE and TRUNCATE in addition to SELECT. Always specify the minimum required: `GRANT SELECT ON patient_records TO analyst_role`.

**5. Forgetting default privileges for future tables**
`GRANT SELECT ON ALL TABLES IN SCHEMA public TO analyst_role` covers existing tables only. Use `ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT ON TABLES TO analyst_role` so new tables automatically inherit correct grants.

**6. search_path injection (Trojan Horse attack)**
If a low-privilege user can create objects in a schema earlier in another user's `search_path`, they can shadow trusted functions. Set `search_path` explicitly for all roles and ensure no untrusted user can write to schemas on the trusted user's path.

---
*sql_methods_library - Samantha McGarrigle*